# Chapter 5 · State Transition and Trace Lab

Predict the final state and first divergence before every run.

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class AccountState:
    balance: float = 0.0
    processed: int = 0
    overdrawn: bool = False

def step(state, event):
    kind, amount = event
    if amount < 0: raise ValueError('amount must be non-negative')
    if kind == 'deposit': new_balance = state.balance + amount
    elif kind in {'withdraw', 'fee'}: new_balance = state.balance - amount
    else: raise ValueError(f'unknown event: {kind}')
    new_state = replace(state, balance=new_balance, processed=state.processed + 1, overdrawn=new_balance < 0)
    return new_state, {'event': event, 'before': state, 'after': new_state}

def run(events, initial=None, max_steps=None):
    state = AccountState() if initial is None else initial
    trace = []
    for index, event in enumerate(events):
        if max_steps is not None and index >= max_steps: break
        state, entry = step(state, event)
        trace.append(entry)
    return state, trace

## Experiment 1 · Same arithmetic, different trace

Predict the final balance and first overdrawn state for both orders.

In [ ]:
a = [('deposit', 100.0), ('withdraw', 80.0), ('fee', 30.0)]
b = [('withdraw', 80.0), ('fee', 30.0), ('deposit', 100.0)]
for name, events in [('A', a), ('B', b)]:
    final, trace = run(events)
    print(name, 'final=', final, 'overdrawn steps=', [i for i, e in enumerate(trace, 1) if e['after'].overdrawn])

## Experiment 2 · Hidden state and resource truncation

Predict how reused state and `max_steps` change the meaning of the output.

In [ ]:
old_state = AccountState(balance=50.0, processed=4)
fresh, _ = run([('deposit', 10.0)])
reused, _ = run([('deposit', 10.0)], initial=old_state)
truncated, trace = run(a, max_steps=2)
print('fresh:', fresh)
print('reused hidden state:', reused)
print('truncated:', truncated, 'steps:', len(trace))

## Experiment 3 · Finite precision

Predict whether repeated binary floating-point addition equals the decimal expectation exactly.

In [ ]:
value = 0.0
for _ in range(10): value += 0.1
print('computed:', repr(value))
print('exactly one:', value == 1.0)

## Reflection

For each experiment, record Input, State, Rule, Order, Resource, Trace, prediction, observation, and the first divergence.